In [ ]:
!pip install -q transformers torch

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import pandas as pd

torch.manual_seed(42)

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Loaded {model_name} on {device}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2 on cpu


In [3]:
def generate_text(prompt, max_length=60, **gen_kwargs):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    output = model.generate(
        input_ids,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        **gen_kwargs
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [4]:
opening_sentence = "The old lighthouse keeper stared out at the storm and"

experiments = [
    {"parameter": "Temperature (low)",   "value": 0.3, "kwargs": dict(do_sample=True, temperature=0.3)},
    {"parameter": "Temperature (high)",  "value": 1.5, "kwargs": dict(do_sample=True, temperature=1.5)},
    {"parameter": "Top-k",               "value": 5,   "kwargs": dict(do_sample=True, top_k=5, temperature=1.0)},
    {"parameter": "Top-p (nucleus)",     "value": 0.9, "kwargs": dict(do_sample=True, top_p=0.9, temperature=1.0)},
    {"parameter": "Repetition penalty",  "value": 2.5, "kwargs": dict(do_sample=True, repetition_penalty=2.5, temperature=1.0)},
]

results = []
for exp in experiments:
    output = generate_text(opening_sentence, **exp["kwargs"])
    results.append({
        "Parameter Changed": exp["parameter"],
        "Value Used": exp["value"],
        "Generated Output": output
    })

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
results_df

,Parameter Changed,Value Used,Generated Output
0,Temperature (low),0.3,"The old lighthouse keeper stared out at the storm and the sky.\n\n""I'm sorry, I'm sorry,"" he said. ""I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry."
1,Temperature (high),1.5,"The old lighthouse keeper stared out at the storm and took it all in. At dawn a single man came under a huge shadow rising by a mile or two. But before anybody knew who this was, the tall man could not make it with any other torch light to reach the new man and there was"
2,Top-k,5.0,"The old lighthouse keeper stared out at the storm and then turned back to look at the sky, then turned back at his own reflection.\n\nHe looked down at the sky and saw the sky was a dark blue.\n\nHe looked up and saw a small, white sphere.\n\nIt"
3,Top-p (nucleus),0.9,"The old lighthouse keeper stared out at the storm and the sky in wonder and wondered if he was going to be saved. He hadn't, but it was so very comforting to see that he had been given a chance. He couldn't do something with the world in his hands, to look at the"
4,Repetition penalty,2.5,"The old lighthouse keeper stared out at the storm and then gave one of his hands a tug as he pulled her into it.\n""I'm sorry,"" she said, tears streaming down Anna's cheeks until they barely touched them anymore; ""We have to get back from this..."" She raised an eyebrow"


In [5]:
for i, row in results_df.iterrows():
    print(f"Experiment {i+1}: {row['Parameter Changed']} = {row['Value Used']}")
    print(f"Output: {row['Generated Output']}")
    print("-" * 80)

Experiment 1: Temperature (low) = 0.3
Output: The old lighthouse keeper stared out at the storm and the sky.

"I'm sorry, I'm sorry," he said. "I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry.
--------------------------------------------------------------------------------
Experiment 2: Temperature (high) = 1.5
Output: The old lighthouse keeper stared out at the storm and took it all in. At dawn a single man came under a huge shadow rising by a mile or two. But before anybody knew who this was, the tall man could not make it with any other torch light to reach the new man and there was
--------------------------------------------------------------------------------
Experiment 3: Top-k = 5.0
Output: The old lighthouse keeper stared out at the storm and then turned back to look at the sky, then turned back at his own reflection.

He looked down at the sky and saw the sky was a dark blue.

He looked up and saw a small, white sphere.

It
-------------

In [6]:
custom_prompt = "Subject: Your order has shipped!\n\nHi there,"

custom_variations = [
    {"label": "Conservative (low temperature)",   "kwargs": dict(do_sample=True, temperature=0.4)},
    {"label": "Balanced (top-p sampling)",         "kwargs": dict(do_sample=True, top_p=0.9, temperature=0.9)},
    {"label": "Creative (higher temperature)",     "kwargs": dict(do_sample=True, temperature=1.2, top_k=50)},
]

custom_results = []
for var in custom_variations:
    output = generate_text(custom_prompt, max_length=70, **var["kwargs"])
    custom_results.append({"Version": var["label"], "Output": output})

custom_df = pd.DataFrame(custom_results)
custom_df

,Version,Output
0,Conservative (low temperature),"Subject: Your order has shipped!\n\nHi there, I have a few questions about the order.\n\n\nI have been waiting for the day that the shipping will be done. I am not sure what to expect, but I am guessing the shipping will be around 3-4 days. I am not sure if the shipping will be delayed or"
1,Balanced (top-p sampling),"Subject: Your order has shipped!\n\nHi there, I'm new to the thread and can't help but feel like I need some help with something.\n\n\nI've read a lot of different sites and some of the reviews seem to say you can't get a good deal.\n\n\nI'm sure I'll get my fix but for"
2,Creative (higher temperature),"Subject: Your order has shipped!\n\nHi there,\n\nThank you for shipping.\n\n\nI am an engineer who worked for several decades on new industrial components I did not know or care about when you contacted me. My concern has been that I received something that was, like, over $500 too much.\n\nAfter researching about"


In [7]:
for i, row in custom_df.iterrows():
    print(f"Version {i+1}: {row['Version']}")
    print(f"Output: {row['Output']}")
    print("-" * 80)

Version 1: Conservative (low temperature)
Output: Subject: Your order has shipped!

Hi there, I have a few questions about the order.


I have been waiting for the day that the shipping will be done. I am not sure what to expect, but I am guessing the shipping will be around 3-4 days. I am not sure if the shipping will be delayed or
--------------------------------------------------------------------------------
Version 2: Balanced (top-p sampling)
Output: Subject: Your order has shipped!

Hi there, I'm new to the thread and can't help but feel like I need some help with something.


I've read a lot of different sites and some of the reviews seem to say you can't get a good deal.


I'm sure I'll get my fix but for
--------------------------------------------------------------------------------
Version 3: Creative (higher temperature)
Output: Subject: Your order has shipped!

Hi there,

Thank you for shipping.


I am an engineer who worked for several decades on new industrial component

In [8]:
extreme_output = generate_text(opening_sentence, do_sample=True, temperature=2.5, max_length=60)
print("Extreme temperature (2.5) output:")
print(extreme_output)

Extreme temperature (2.5) output:
The old lighthouse keeper stared out at the storm and watched that dark reflection of its shining beacon into nothing else of all places there to lie dormant and not awake when it caught out into the cold black heavens in silence from her view with his little blue blibbs that stuck, just now starting up against


Across the five experiments, temperature had the clearest effect on output style: low temperature (0.3) produced safe, repetitive, and highly predictable text, while high temperature (1.5) produced more varied and creative — but occasionally less coherent — continuations. Top-k sampling kept the output focused by restricting choices to the 5 most likely next words, giving a middle ground between predictability and variety. Top-p (nucleus) sampling felt the most natural overall, since it adapts the candidate pool size to the model's actual confidence at each step rather than using a fixed cutoff. Repetition penalty noticeably reduced the looping and repeated phrases that plain sampling sometimes falls into, though at very high values it occasionally forced slightly awkward word choices. Pushing temperature to an extreme (2.5) produced largely incoherent, almost random text — because at that setting the model's probability distribution over next words becomes nearly flat, so it ends up picking very unlikely tokens almost as often as likely ones, breaking the sentence's logical flow. Overall, temperature and top-p had the strongest influence on the creativity/coherence trade-off, while repetition penalty was most useful for controlling redundancy rather than randomness.